# 623 Stride v25: dual-context hurdle LSTM sweep

This notebook reuses the verified v23 input bytes and runs the five fixed capacities once. Each model sees only raw PC/address input, splits H between global and exact-PC-local LSTMs, learns a ZERO/POSITIVE hurdle and positive count, and emits ordered unique direct targets. Teacher actions are labels only.


In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
from google.colab import userdata
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), f'Select an A100 runtime, observed {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}'
torch.set_float32_matmul_precision('highest')
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)
REPO='/content/cache_arch'
TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
os.chmod(ASKPASS,0o700)
env=os.environ.copy()
env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else:
        subprocess.run(['git','-C',REPO,'switch','main'],check=True,env=env)
        subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
EXP=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride'
SCRIPT=f'{EXP}/python/train_and_offline_infer.py'
CONTRACT_SCRIPT=f'{EXP}/python/model_contract.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--describe-model-points'],text=True))
TRAINER_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
assert MODEL_CONTRACT==TRAINER_CONTRACT
RUN_ID=MODEL_CONTRACT['run_id']
TRAINING=MODEL_CONTRACT['training_config']
TRACE=MODEL_CONTRACT['trace']
POLICY=MODEL_CONTRACT['policy']
DRIVE_BASE=pathlib.Path(f'/content/drive/MyDrive/cache_prefetch_623_{POLICY}')
INPUT_CACHE=DRIVE_BASE/'input_cache'
DRIVE_RUN=DRIVE_BASE/'runs'/RUN_ID
INPUT_DIR=pathlib.Path(f'/content/{RUN_ID}_colab_input')
CANONICAL_INPUT=INPUT_CACHE/f'{RUN_ID}.colab_input.tar.gz'
INPUT_CACHE.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,f'{REPO}/formal_NN_training/common')
import split_colab_archive as transfer

def extract_input(archive):
    if INPUT_DIR.exists():
        shutil.rmtree(INPUT_DIR)
    transfer.safe_extract_tar_gz(archive,INPUT_DIR)
    verified=transfer.validate_sha256sums(INPUT_DIR)
    assert verified
    return verified

if CANONICAL_INPUT.is_file():
    try:
        VERIFIED_INPUT=extract_input(CANONICAL_INPUT)
        print('reused input from Drive',CANONICAL_INPUT)
    except Exception as exc:
        print('Drive cache invalid; upload the input again',repr(exc))
        CANONICAL_INPUT.unlink(missing_ok=True)

if not CANONICAL_INPUT.is_file():
    print('Select the v23 Stride .colab_input.tar.gz')
    uploaded=files.upload()
    assert len(uploaded)==1
    name,payload=next(iter(uploaded.items()))
    assert name.endswith('.colab_input.tar.gz')
    temporary=pathlib.Path('/content')/name
    temporary.write_bytes(payload)
    VERIFIED_INPUT=extract_input(temporary)
    shutil.copyfile(temporary,CANONICAL_INPUT)

print('verified input payload files',len(VERIFIED_INPUT))

# Every Run all is a clean, one-pass experiment.  The immutable input cache is
# outside the run directory, so stale models/logs/archives cannot be resumed.
if DRIVE_RUN.exists():
    shutil.rmtree(DRIVE_RUN)
DRIVE_RUN.mkdir(parents=True)
OUTPUT_ROOT=DRIVE_RUN/'colab_output'
OUTPUT_ARCHIVE=DRIVE_RUN/f'{RUN_ID}.colab_output.tar.gz'
OUTPUT_ARCHIVE.unlink(missing_ok=True)


In [ ]:
ROLES=('train','guard','eval')
INPUTS={role:{
    'stream':str(INPUT_DIR/f'{TRACE}.{POLICY}.{role}_stream.csv.gz'),
    'candidates':str(INPUT_DIR/f'{TRACE}.{POLICY}.{role}_candidates.csv.gz'),
} for role in ROLES}
for items in INPUTS.values():
    for path in items.values():
        assert pathlib.Path(path).is_file(),path
VALIDATOR=f'{EXP}/python/validate_collected_inputs.py'
VALIDATED_MANIFEST=DRIVE_RUN/'validated_collection_manifest.json'
CHILD_ENV=os.environ.copy()
CHILD_ENV['PYTHONUNBUFFERED']='1'
subprocess.run([
    sys.executable,VALIDATOR,'--input-dir',str(INPUT_DIR),
    '--manifest-out',str(VALIDATED_MANIFEST)
],check=True,env=CHILD_ENV)
manifest=json.loads(VALIDATED_MANIFEST.read_text())
assert manifest['status']=='PASS'
assert manifest['source_decision_effective_external_input']==MODEL_CONTRACT['external_input_fields']
assert manifest['dual_context_core_used'] is True
assert manifest['categorical_count_head_used'] is True
assert manifest['positive_only_categorical_count_head_used'] is True
assert manifest['hurdle_head_used'] is True
assert manifest['count_zero_is_implicit_hurdle'] is True
assert manifest['loss_class_reweighting_used'] is False
assert manifest['decoded_target_projection_or_mutation_used'] is False
assert manifest['delta_token_head_used'] is False
assert manifest['delta_vocabulary_used'] is False
assert manifest['delta_escape_head_used'] is False
assert manifest['rank_delta_payload_bits']==58
print('input and v25 contract PASS')


In [ ]:
LOCAL_OUTPUT=pathlib.Path(f'/content/{RUN_ID}_colab_output')
if LOCAL_OUTPUT.exists():
    shutil.rmtree(LOCAL_OUTPUT)
LOCAL_OUTPUT.mkdir()
shutil.copy2(VALIDATED_MANIFEST,LOCAL_OUTPUT/'validated_collection_manifest.json')
LOG_DIR=DRIVE_RUN/'trainer_logs'
LOG_DIR.mkdir(parents=True,exist_ok=True)

def run_streamed(cmd,log_path):
    print('trainer command',json.dumps(cmd),flush=True)
    with pathlib.Path(log_path).open('w',encoding='utf-8',buffering=1) as log:
        log.write('command='+json.dumps(cmd)+'\n')
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=CHILD_ENV)
        assert process.stdout is not None
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        returncode=process.wait()
        log.write(f'\nreturncode={returncode}\n')
    if returncode:
        raise subprocess.CalledProcessError(returncode,cmd)

POINTS=sorted(MODEL_CONTRACT['points'],key=lambda point:point['model_size'])
assert [point['model_size'] for point in POINTS]==[8,16,32,64,128]
SWEEP=[]
for point in POINTS:
    out=LOCAL_OUTPUT/point['model_tag']
    cmd=[sys.executable,SCRIPT,'--policy',POLICY]
    for role in ROLES:
        cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
    cmd += [
        '--out-dir',str(out),'--model-family','lstm',
        '--model-size',str(point['model_size']),
        '--pair-id',point['architecture_pair_id'],
        '--device','cuda','--seed',str(TRAINING['seed']),
        '--epochs',str(TRAINING['epochs']),
        '--chunk-len',str(TRAINING['chunk_len']),
        '--accumulate-chunks',str(TRAINING['accumulate_chunks']),
        '--learning-rate',str(TRAINING['learning_rate']),
    ]
    log_path=LOG_DIR/f"{point['model_tag']}.stdout_stderr.log"
    run_streamed(cmd,log_path)
    shutil.copy2(log_path,out/'trainer.stdout_stderr.log')
    meta=json.loads((out/'run_metadata.json').read_text())
    expected={
        'run_id':RUN_ID,'model_tag':point['model_tag'],
        'model_size':point['model_size'],
        'architecture_pair_id':point['architecture_pair_id'],
        'matched_normal_prefetcher':POLICY,
        'teacher_actions_are_model_inputs':False,
        'normal_policy_outputs_used_as_model_inputs':False,
        'normal_policy_candidates_used_as_model_inputs':False,
        'normal_policy_private_state_used_as_model_inputs':False,
        'normal_policy_request_rate_used_as_budget':False,
        'categorical_count_head_used':True,
        'positive_only_categorical_count_head_used':True,
        'count_zero_is_implicit_hurdle':True,
        'count_regression_used':False,'log_count_used':False,
        'hurdle_head_used':True,'stop_padding_used':False,
        'loss_class_reweighting_used':False,
        'decode_prior_correction_used':False,
        'probability_threshold_used':False,
        'inference_policy_hardcodes_used':False,
        'dual_context_core_used':True,
        'decoded_target_projection_or_mutation_used':False,
        'delta_token_head_used':False,
        'delta_vocabulary_used':False,
        'delta_escape_head_used':False,
        'rank_delta_payload_bits':58,
        'original_guard_used_for_checkpoint_selection':False,
        'evaluation_used_for_checkpoint_selection':False,
        'weights_retrained':True,'checkpoint_reused':False,
        'final_retrained_from_scratch':True,
    }
    bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}
    assert not bad,bad
    assert meta['model_point_contract']==MODEL_CONTRACT
    assert meta['oracle_diagnostics_replayed'] is False
    SWEEP.append({
        'model_tag':point['model_tag'],
        'model_size':point['model_size'],
        'parameter_count':meta['parameter_count'],
        'selected_epoch':meta['selected_epoch'],
        'complete_validation_nll':meta['selected_blocked_validation'],
        'final_retrain_epochs':meta['final_retrain_epochs'],
        'heldout_behavior_metrics':meta['heldout_behavior_metrics'],
        'oracle_diagnostics':meta['oracle_diagnostics'],
    })

sweep_manifest={
    'status':'PASS',
    'run_id':RUN_ID,'trace':TRACE,'policy':POLICY,
    'model_revision':MODEL_CONTRACT['model_revision'],
    'decoder_revision':MODEL_CONTRACT['decoder_revision'],
    'training_config':TRAINING,'model_contract':MODEL_CONTRACT,
    'fresh_input_validation_manifest':VALIDATED_MANIFEST.name,
    'points':SWEEP,
}
(LOCAL_OUTPUT/'sweep_manifest.json').write_text(json.dumps(sweep_manifest,indent=2,sort_keys=True)+'\n')
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
print(json.dumps(SWEEP,indent=2))


In [ ]:
temporary=OUTPUT_ARCHIVE.with_name(f'.{OUTPUT_ARCHIVE.name}.writing')
temporary.unlink(missing_ok=True)
with tarfile.open(temporary,'w:gz') as archive:
    for item in sorted(OUTPUT_ROOT.iterdir(),key=lambda path:path.name):
        archive.add(item,arcname=item.name)
with tarfile.open(temporary,'r:gz') as archive:
    assert archive.getmembers()
os.replace(temporary,OUTPUT_ARCHIVE)
print('saved',OUTPUT_ARCHIVE,OUTPUT_ARCHIVE.stat().st_size,'bytes')
files.download(str(OUTPUT_ARCHIVE))


The archive contains five independently trained v25 models, their final action lists, two-phase training histories, complete validation-NLL evidence, phase-shift GUARD audits, and diagnosis-only oracle decompositions. Oracle variants are not replayed and cannot be reported as neural wins.
